#Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Import Datset

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/HazemNegm/GDG-Final-Project/968668ebe662b3d9f3ef228b7b6348d8cc60995c/Raw%20Data/global_tech_startups_2026.csv")

#Dashboard 1 (Overview)

In [3]:
"""
Dashboard 1 - Overview
======================
Purpose: high-level snapshot of startup size, geography, sectors,
funding stages, AI adoption, and historical layoffs.

Dataset : cleaned_Data (1).csv  (Global Tech Startups 2026)
Financial columns are in USD millions.
Run in Jupyter/Colab (needs: pandas, plotly, ipython).
"""

import pandas as pd
import plotly.express as px

# display() works in Jupyter; falls back to print() in plain scripts
try:
    from IPython.display import display
except ImportError:
    display = print

# ---------------------------------------------------------------
# 1) Load the dataset
# ---------------------------------------------------------------
df = pd.read_csv("https://raw.githubusercontent.com/HazemNegm/GDG-Final-Project/968668ebe662b3d9f3ef228b7b6348d8cc60995c/Raw%20Data/global_tech_startups_2026.csv")

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))

# ---------------------------------------------------------------
# 2) Data-quality checks before calculating KPIs
# ---------------------------------------------------------------
# The dataset may contain repeated Company_ID values.
# Validate them before counting distinct companies.
duplicate_ids = df.loc[df["Company_ID"].duplicated(keep=False), "Company_ID"].unique()
print("\nRepeated Company_ID values:", list(duplicate_ids))

# AI_Adoption_Level already keeps missing values as 'Unknown'.

# ---------------------------------------------------------------
# 3) KPI cards
# ---------------------------------------------------------------
kpis = {
    "Total Companies (distinct)": df["Company_ID"].nunique(),
    "Total Domains": df["Domain"].nunique(),
    "Distinct Countries": df["Country"].nunique(),
    "Distinct Cities": df["City"].nunique(),
    "Total Funding (USD M)": round(df["Total_Funding_USD_Millions"].sum(), 2),
    "Total Valuation (USD M)": round(df["Valuation_USD_Millions"].sum(), 2),
    "Total Revenue ARR (USD M)": round(df["Revenue_ARR_Millions"].sum(), 2),
    "Total Layoffs (2024-2025)": int(df["Layoffs_2024_2025"].sum()),
    "Average Runway (Months)": round(df["Runway_Months_2024"].mean(), 2),
    "Median Runway (Months)": round(df["Runway_Months_2024"].median(), 2),
}

kpi_df = pd.DataFrame(list(kpis.items()), columns=["KPI", "Value"])
display(kpi_df)

# ---------------------------------------------------------------
# 4) Business questions
# ---------------------------------------------------------------

# --- Q1: How many distinct startups are represented? ---
# Already shown in the KPI card above as "Total Companies (distinct)".

# --- Q2: How are startups distributed across domains? ---
domain_counts = df["Domain"].value_counts().reset_index()
domain_counts.columns = ["Domain", "Companies"]

display(domain_counts)

fig = px.bar(
    domain_counts, x="Domain", y="Companies",
    title="Q2 - Startups by Domain", text="Companies",
    color_discrete_sequence=["#636EFA"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Domain", yaxis_title="Number of Startups")
fig.show()

# --- Q3: How are startups distributed by founding year? ---
founding_year_counts = df["Founding_Year"].value_counts().sort_index().reset_index()
founding_year_counts.columns = ["Founding_Year", "Companies"]

display(founding_year_counts)

fig = px.line(
    founding_year_counts, x="Founding_Year", y="Companies",
    title="Q3 - Startups Founded by Year", markers=True,
    color_discrete_sequence=["purple"],
)
fig.update_layout(xaxis_title="Founding Year", yaxis_title="Number of Startups")
fig.show()

# --- Q4: Which countries and cities contain the most startups? ---
country_counts = df["Country"].value_counts().reset_index()
country_counts.columns = ["Country", "Companies"]

display(country_counts)

fig = px.bar(
    country_counts, x="Country", y="Companies",
    title="Q4 - Startups by Country", text="Companies",
    color_discrete_sequence=["#00CC96"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Country", yaxis_title="Number of Startups")
fig.show()

# Map (choropleth): map country names to ISO codes for plotting
country_iso = {
    "USA": "USA", "China": "CHN", "India": "IND", "UK": "GBR",
    "Canada": "CAN", "Germany": "DEU", "France": "FRA",
    "Israel": "ISR", "Japan": "JPN", "Singapore": "SGP",
}
country_counts["ISO"] = country_counts["Country"].map(country_iso)

fig = px.choropleth(
    country_counts, locations="ISO", color="Companies",
    title="Q4 - Startups by Country (Map)",
    color_continuous_scale="Viridis", hover_name="Country",
)
fig.show()

city_counts = df["City"].value_counts().head(10).reset_index()
city_counts.columns = ["City", "Companies"]

display(city_counts)

fig = px.bar(
    city_counts, x="City", y="Companies",
    title="Q4 - Top 10 Cities by Startups", text="Companies",
    color_discrete_sequence=["#AB63FA"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="City", yaxis_title="Number of Startups")
fig.show()

# --- Q5: How are startups distributed across funding stages? ---
stage_counts = df["Funding_Stage"].value_counts().reset_index()
stage_counts.columns = ["Funding_Stage", "Companies"]

display(stage_counts)

fig = px.pie(
    stage_counts, names="Funding_Stage", values="Companies",
    title="Q5 - Startups by Funding Stage", hole=0.4,
)
fig.show()

fig = px.bar(
    stage_counts, x="Funding_Stage", y="Companies",
    title="Q5 - Startups by Funding Stage", text="Companies",
    color_discrete_sequence=["#FFA15A"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Funding Stage", yaxis_title="Number of Startups")
fig.show()

# --- Q6: How are startups distributed by AI adoption level? ---
ai_counts = df["AI_Adoption_Level"].value_counts().reset_index()
ai_counts.columns = ["AI_Adoption_Level", "Companies"]

display(ai_counts)

fig = px.bar(
    ai_counts, x="AI_Adoption_Level", y="Companies",
    title="Q6 - Startups by AI Adoption Level", text="Companies",
    color_discrete_sequence=["#19D3F3"],
)
fig.update_layout(xaxis_title="AI Adoption Level", yaxis_title="Number of Startups")
fig.show()

# --- Q7: What share of companies recorded layoffs during 2024-2025? ---
df["Has_Layoffs"] = df["Layoffs_2024_2025"] > 0

companies_with_layoffs = df["Has_Layoffs"].sum()
layoff_share = df["Has_Layoffs"].mean() * 100

print(f"\nCompanies with layoffs: {companies_with_layoffs:,} / {len(df):,}")
print(f"Share of companies with layoffs: {layoff_share:.1f}%")

layoff_counts = df["Has_Layoffs"].value_counts().reset_index()
layoff_counts.columns = ["Has_Layoffs", "Companies"]
layoff_counts["Has_Layoffs"] = layoff_counts["Has_Layoffs"].map(
    {True: "With Layoffs", False: "Without Layoffs"}
)

fig = px.bar(
    layoff_counts, x="Has_Layoffs", y="Companies",
    title="Q7 - Companies With vs Without Layoffs (2024-2025)",
    text="Companies", color="Has_Layoffs",
)
fig.update_layout(xaxis_title="", yaxis_title="Number of Startups", showlegend=False)
fig.show()

# --- Q8: How do historical layoffs vary by company size? ---
# Company size is approximated with Peak_Headcount_2023.
fig = px.scatter(
    df, x="Peak_Headcount_2023", y="Layoffs_2024_2025",
    title="Q8 - Layoffs vs Company Size",
    labels={
        "Peak_Headcount_2023": "Peak Headcount (2023)",
        "Layoffs_2024_2025": "Layoffs (2024-2025)",
    },
    opacity=0.4, color_discrete_sequence=["#EF553B"],
)
fig.show()

# Grouped view by company-size bucket
df["Size_Bucket"] = pd.cut(
    df["Peak_Headcount_2023"],
    bins=[0, 25, 50, 100, 250, 1000, 50000],
    labels=["1-25", "26-50", "51-100", "101-250", "251-1000", "1000+"],
)

size_layoffs = df.groupby("Size_Bucket", observed=True).agg(
    companies=("Company_ID", "count"),
    total_layoffs=("Layoffs_2024_2025", "sum"),
    layoff_rate_pct=("Has_Layoffs", "mean"),
).reset_index()
size_layoffs["layoff_rate_pct"] = size_layoffs["layoff_rate_pct"] * 100

display(size_layoffs)

fig = px.bar(
    size_layoffs, x="Size_Bucket", y="total_layoffs",
    title="Q8 - Total Layoffs by Company Size Bucket", text="total_layoffs",
    color_discrete_sequence=["#EF553B"],
)
fig.update_layout(xaxis_title="Company Size (Headcount)", yaxis_title="Total Layoffs (2024-2025)")
fig.show()

Dataset shape: (25000, 17)
Columns: ['Company_ID', 'Domain', 'Founding_Year', 'Country', 'City', 'Funding_Stage', 'Total_Funding_USD_Millions', 'Valuation_USD_Millions', 'Revenue_ARR_Millions', 'Monthly_Burn_Rate_Millions', 'Runway_Months_2024', 'Peak_Headcount_2023', 'Layoffs_2024_2025', 'Current_Headcount_2026', 'Investor_Tier', 'AI_Adoption_Level', 'Acquisition_Status']

Repeated Company_ID values: ['7791E581']


,KPI,Value
0,Total Companies (distinct),24999.00
1,Total Domains,10.00
2,Distinct Countries,10.00
3,Distinct Cities,29.00
4,Total Funding (USD M),1490777.46
5,Total Valuation (USD M),11602503.03
6,Total Revenue ARR (USD M),832522.26
7,Total Layoffs (2024-2025),789083.00
8,Average Runway (Months),12.19
9,Median Runway (Months),10.60


,Domain,Companies
0,Generative AI,4958
1,HealthTech,3760
2,FinTech,3060
3,ClimateTech,2507
4,Robotics,2468
5,Cybersecurity,2466
6,Web3/Crypto,2047
7,SpaceTech,1252
8,Autonomous Vehicles,1248
9,EdTech,1234


,Founding_Year,Companies
0,2012,492
1,2013,761
2,2014,1236
3,2015,1317
4,2016,1239
5,2017,1293
6,2018,2592
7,2019,3685
8,2020,5047
9,2021,3671


,Country,Companies
0,USA,11178
1,China,3755
2,India,2560
3,UK,1998
4,Israel,1277
5,Germany,1270
6,Canada,1057
7,France,729
8,Singapore,698
9,Japan,478


,City,Companies
0,Boston,1901
1,New York,1893
2,Seattle,1886
3,San Francisco,1878
4,Miami,1844
5,Austin,1776
6,Shanghai,966
7,Hangzhou,941
8,Shenzhen,927
9,Beijing,921


,Funding_Stage,Companies
0,Seed,8859
1,Series A,6154
2,Series B,5014
3,Series C+,2996
4,Pre-IPO,1257
5,Post-IPO,720


,AI_Adoption_Level,Companies
0,High,9814
1,Medium,7512
2,Low,5082



Companies with layoffs: 9,710 / 25,000
Share of companies with layoffs: 38.8%


,Size_Bucket,companies,total_layoffs,layoff_rate_pct
0,1-25,4738,14296,37.821866
1,26-50,8805,58083,36.104486
2,51-100,3645,33317,34.458162
3,101-250,3716,109517,47.012917
4,251-1000,3098,209573,42.543577
5,1000+,998,364297,41.883768


#Dashboard 2 (Investment)

In [4]:
"""
Dashboard 2 - Investment
========================
Purpose: identify where capital is concentrated, how valuations vary,
and which investment and liquidity indicators are associated with
historical layoffs.

Dataset : cleaned_Data (1).csv  (Global Tech Startups 2026)
Financial columns are in USD millions.
Run in Jupyter/Colab (needs: pandas, plotly, ipython).
"""

import pandas as pd
import plotly.express as px

try:
    from IPython.display import display
except ImportError:
    display = print

# ---------------------------------------------------------------
# 1) Load the dataset
# ---------------------------------------------------------------
df = pd.read_csv("https://raw.githubusercontent.com/HazemNegm/GDG-Final-Project/968668ebe662b3d9f3ef228b7b6348d8cc60995c/Raw%20Data/global_tech_startups_2026.csv")

# Flag companies with any layoffs in 2024-2025
df["Has_Layoffs"] = df["Layoffs_2024_2025"] > 0

# ---------------------------------------------------------------
# 2) KPI cards
# ---------------------------------------------------------------
kpis = {
    "Total Funding (USD M)": round(df["Total_Funding_USD_Millions"].sum(), 2),
    "Total Valuation (USD M)": round(df["Valuation_USD_Millions"].sum(), 2),
    "Average Funding per Company (USD M)": round(df["Total_Funding_USD_Millions"].mean(), 2),
    "Median Funding per Company (USD M)": round(df["Total_Funding_USD_Millions"].median(), 2),
    "Average Runway (Months)": round(df["Runway_Months_2024"].mean(), 2),
    "Median Runway (Months)": round(df["Runway_Months_2024"].median(), 2),
    "Layoff Company Rate (%)": round(df["Has_Layoffs"].mean() * 100, 2),
}

kpi_df = pd.DataFrame(list(kpis.items()), columns=["KPI", "Value"])
display(kpi_df)

# Funding by stage (KPI): totals and company counts
funding_by_stage = df.groupby("Funding_Stage", sort=False).agg(
    total_funding=("Total_Funding_USD_Millions", "sum"),
    companies=("Company_ID", "nunique"),
).reset_index()
display(funding_by_stage)

# ---------------------------------------------------------------
# 3) Business questions
# ---------------------------------------------------------------

# --- Q1: Which domains receive the most total funding? ---
funding_by_domain = (
    df.groupby("Domain", sort=False)
    .agg(total_funding=("Total_Funding_USD_Millions", "sum"))
    .reset_index()
    .sort_values("total_funding", ascending=False)
)
display(funding_by_domain)

fig = px.bar(
    funding_by_domain, x="Domain", y="total_funding",
    title="Q1 - Total Funding by Domain", text="total_funding",
    color_discrete_sequence=["#636EFA"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Domain", yaxis_title="Total Funding (USD M)")
fig.show()

# --- Q2: Which countries receive the most total funding? ---
funding_by_country = (
    df.groupby("Country", sort=False)
    .agg(total_funding=("Total_Funding_USD_Millions", "sum"))
    .reset_index()
    .sort_values("total_funding", ascending=False)
)
display(funding_by_country)

fig = px.bar(
    funding_by_country, x="Country", y="total_funding",
    title="Q2 - Total Funding by Country", text="total_funding",
    color_discrete_sequence=["#00CC96"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Country", yaxis_title="Total Funding (USD M)")
fig.show()

# --- Q3: Which cities receive the most total funding? ---
funding_by_city = (
    df.groupby("City", sort=False)
    .agg(total_funding=("Total_Funding_USD_Millions", "sum"))
    .reset_index()
    .sort_values("total_funding", ascending=False)
    .head(10)
)
display(funding_by_city)

fig = px.bar(
    funding_by_city, x="City", y="total_funding",
    title="Q3 - Total Funding by City (Top 10)", text="total_funding",
    color_discrete_sequence=["#AB63FA"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="City", yaxis_title="Total Funding (USD M)")
fig.show()

# --- Q4: How does funding vary by funding stage? ---
funding_by_stage = (
    df.groupby("Funding_Stage", sort=False)
    .agg(
        total_funding=("Total_Funding_USD_Millions", "sum"),
        companies=("Company_ID", "nunique"),
    )
    .reset_index()
    .sort_values("total_funding", ascending=False)
)
display(funding_by_stage)

fig = px.bar(
    funding_by_stage, x="Funding_Stage", y="total_funding",
    title="Q4 - Total Funding by Funding Stage", text="total_funding",
    color_discrete_sequence=["#FFA15A"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Funding Stage", yaxis_title="Total Funding (USD M)")
fig.show()

# --- Q5: How does valuation vary by country and city? ---
valuation_by_country = (
    df.groupby("Country", sort=False)
    .agg(total_valuation=("Valuation_USD_Millions", "sum"))
    .reset_index()
    .sort_values("total_valuation", ascending=False)
)
display(valuation_by_country)

fig = px.bar(
    valuation_by_country, x="Country", y="total_valuation",
    title="Q5 - Total Valuation by Country", text="total_valuation",
    color_discrete_sequence=["#19D3F3"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Country", yaxis_title="Total Valuation (USD M)")
fig.show()

valuation_by_city = (
    df.groupby("City", sort=False)
    .agg(total_valuation=("Valuation_USD_Millions", "sum"))
    .reset_index()
    .sort_values("total_valuation", ascending=False)
    .head(10)
)
display(valuation_by_city)

fig = px.bar(
    valuation_by_city, x="City", y="total_valuation",
    title="Q5 - Total Valuation by City (Top 10)", text="total_valuation",
    color_discrete_sequence=["#19D3F3"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="City", yaxis_title="Total Valuation (USD M)")
fig.show()

# --- Q6: Is higher funding associated with higher valuation? ---
corr_fv = df["Total_Funding_USD_Millions"].corr(df["Valuation_USD_Millions"])
print(f"\nCorrelation (funding vs valuation): {corr_fv:.3f}")

fig = px.scatter(
    df, x="Total_Funding_USD_Millions", y="Valuation_USD_Millions",
    title="Q6 - Funding vs Valuation", opacity=0.35,
    labels={
        "Total_Funding_USD_Millions": "Total Funding (USD M)",
        "Valuation_USD_Millions": "Valuation (USD M)",
    },
    color_discrete_sequence=["#636EFA"],
)
fig.show()

# --- Q7: How do funding and valuation vary by investor tier? ---
tier_summary = (
    df.groupby("Investor_Tier", sort=False)
    .agg(
        companies=("Company_ID", "nunique"),
        total_funding=("Total_Funding_USD_Millions", "sum"),
        total_valuation=("Valuation_USD_Millions", "sum"),
    )
    .reset_index()
    .sort_values("total_funding", ascending=False)
)
display(tier_summary)

fig = px.bar(
    tier_summary, x="Investor_Tier", y="total_funding",
    title="Q7 - Total Funding by Investor Tier", text="total_funding",
    color_discrete_sequence=["#00CC96"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Investor Tier", yaxis_title="Total Funding (USD M)")
fig.show()

# --- Q8: How are funding and runway associated with layoff rates? ---
risk_by_domain = df.groupby("Domain", sort=False).agg(
    companies=("Company_ID", "nunique"),
    total_funding=("Total_Funding_USD_Millions", "sum"),
    avg_runway=("Runway_Months_2024", "mean"),
    layoff_rate_pct=("Has_Layoffs", "mean"),
).reset_index()
risk_by_domain["layoff_rate_pct"] = risk_by_domain["layoff_rate_pct"] * 100

display(risk_by_domain.sort_values("layoff_rate_pct", ascending=False))

fig = px.scatter(
    risk_by_domain, x="avg_runway", y="layoff_rate_pct",
    size="total_funding", color="Domain", hover_name="Domain",
    title="Q8 - Runway vs Layoff Rate by Domain (bubble = total funding)",
    labels={
        "avg_runway": "Average Runway (Months)",
        "layoff_rate_pct": "Layoff Company Rate (%)",
    },
)
fig.show()

# --- Q9: Do historical layoff rates differ across funding stages? ---
stage_layoff = (
    df.groupby(["Funding_Stage", "Has_Layoffs"], sort=False)
    .size()
    .reset_index(name="companies")
)
stage_layoff["pct"] = stage_layoff.groupby("Funding_Stage")["companies"].transform(
    lambda x: x / x.sum() * 100
)
stage_layoff["Has_Layoffs"] = stage_layoff["Has_Layoffs"].map(
    {True: "With Layoffs", False: "Without Layoffs"}
)

fig = px.bar(
    stage_layoff, x="Funding_Stage", y="pct", color="Has_Layoffs",
    title="Q9 - Layoff Company Share by Funding Stage (100% stacked)",
    barmode="stack", text="pct",
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Funding Stage", yaxis_title="Share of Companies (%)")
fig.show()

# --- Q10: Which domains combine high funding with high layoff rates? ---
fig = px.scatter(
    risk_by_domain, x="total_funding", y="layoff_rate_pct", text="Domain",
    title="Q10 - High Funding vs High Layoff Rate by Domain",
    labels={
        "total_funding": "Total Funding (USD M)",
        "layoff_rate_pct": "Layoff Company Rate (%)",
    },
    color_discrete_sequence=["#EF553B"],
)
fig.update_traces(textposition="top center")
fig.show()

,KPI,Value
0,Total Funding (USD M),1490777.46
1,Total Valuation (USD M),11602503.03
2,Average Funding per Company (USD M),59.63
3,Median Funding per Company (USD M),8.46
4,Average Runway (Months),12.19
5,Median Runway (Months),10.60
6,Layoff Company Rate (%),38.84


,Funding_Stage,total_funding,companies
0,Seed,23938.52,8858
1,Pre-IPO,361750.60,1257
2,Series A,82530.37,6154
3,Series B,200242.09,5014
4,Series C+,326151.08,2996
5,Post-IPO,496164.80,720


,Domain,total_funding
4,Generative AI,567684.22
0,HealthTech,162674.79
3,FinTech,127543.21
8,SpaceTech,122450.19
1,Web3/Crypto,108547.15
7,Robotics,103468.35
6,ClimateTech,99591.54
2,Cybersecurity,90135.28
9,Autonomous Vehicles,58220.25
5,EdTech,50462.48


,Country,total_funding
2,USA,691264.80
1,China,210420.30
3,India,143859.37
4,UK,111960.08
6,Germany,83605.93
7,Israel,76053.24
0,Canada,53741.58
5,Singapore,46738.20
9,France,42262.51
8,Japan,30871.45


,City,total_funding
16,Boston,130659.58
7,Miami,126261.67
2,San Francisco,117278.81
18,Austin,109846.56
9,Seattle,106546.50
4,New York,100671.68
5,Shanghai,60030.08
15,Hangzhou,54028.63
23,Shenzhen,49646.51
8,Singapore,46738.20


...

,Country,total_valuation
2,USA,5488733.63
1,China,1636658.73
3,India,1106878.10
4,UK,885847.75
6,Germany,600850.21
7,Israel,565127.18
0,Canada,416504.39
5,Singapore,342288.67
9,France,315427.54
8,Japan,244186.83


,City,total_valuation
16,Boston,1057458.80
7,Miami,931634.87
2,San Francisco,929990.05
18,Austin,903093.98
9,Seattle,838642.63
4,New York,827913.30
5,Shanghai,459656.01
23,Shenzhen,409691.33
15,Hangzhou,392601.77
1,Beijing,374709.62



Correlation (funding vs valuation): 0.945


,Investor_Tier,companies,total_funding,total_valuation
0,Tier 3 (Angel/Seed),9964,559176.59,4369407.63
1,Tier 2 (Mid-size VC),7477,402021.27,3118265.79
3,"Tier 1 (e.g. Sequoia, a16z)",2544,301560.56,2330992.32
2,Corporate VC (e.g. Google Ventures),3839,228019.04,1753848.53
4,Bootstrapped,1175,0.00,29988.76


,Domain,companies,total_funding,avg_runway,layoff_rate_pct
1,Web3/Crypto,2047,108547.15,12.195554,84.953591
9,Autonomous Vehicles,1248,58220.25,10.894311,40.144231
5,EdTech,1234,50462.48,10.768233,39.870340
6,ClimateTech,2507,99591.54,10.824013,39.050658
0,HealthTech,3760,162674.79,11.689495,38.670213
7,Robotics,2468,103468.35,11.358306,38.006483
2,Cybersecurity,2466,90135.28,10.845134,37.793998
3,FinTech,3060,127543.21,11.344020,37.712418
8,SpaceTech,1252,122450.19,14.092412,37.699681
4,Generative AI,4958,567684.22,15.057745,21.157725


#Dashboard 3 (Revenue)

In [5]:
"""
Dashboard 3 - Revenue
=====================
Purpose: compare recurring revenue across domains and markets, and
explore how revenue relates to AI adoption, funding, cash burn, and
historical layoffs.

Dataset : cleaned_Data (1).csv  (Global Tech Startups 2026)
Financial columns are in USD millions.
Run in Jupyter/Colab (needs: pandas, plotly, ipython).
"""

import pandas as pd
import plotly.express as px

try:
    from IPython.display import display
except ImportError:
    display = print

# ---------------------------------------------------------------
# 1) Load the dataset
# ---------------------------------------------------------------
df = pd.read_csv("https://raw.githubusercontent.com/HazemNegm/GDG-Final-Project/968668ebe662b3d9f3ef228b7b6348d8cc60995c/Raw%20Data/global_tech_startups_2026.csv")

# Flag companies with any layoffs in 2024-2025
df["Has_Layoffs"] = df["Layoffs_2024_2025"] > 0

# Derived revenue-sustainability columns
df["Monthly_ARR_Proxy"] = df["Revenue_ARR_Millions"] / 12
df["ARR_Minus_Burn"] = df["Monthly_ARR_Proxy"] - df["Monthly_Burn_Rate_Millions"]
df["Revenue_To_Burn"] = df["Revenue_ARR_Millions"] / (12 * df["Monthly_Burn_Rate_Millions"])

# ---------------------------------------------------------------
# 2) KPI cards
# ---------------------------------------------------------------
total_arr = df["Revenue_ARR_Millions"].sum()
total_burn = df["Monthly_Burn_Rate_Millions"].sum()

kpis = {
    "Total Revenue ARR (USD M)": round(total_arr, 2),
    "Average ARR per Company (USD M)": round(df["Revenue_ARR_Millions"].mean(), 2),
    "Median ARR per Company (USD M)": round(df["Revenue_ARR_Millions"].median(), 2),
    "Total Monthly Burn (USD M/mo)": round(total_burn, 2),
    "Revenue-to-Burn Ratio (total)": round(total_arr / (12 * total_burn), 2),
    "Total Monthly ARR Proxy Minus Burn (USD M)": round(df["ARR_Minus_Burn"].sum(), 2),
    "Layoff Company Rate (%)": round(df["Has_Layoffs"].mean() * 100, 2),
}

kpi_df = pd.DataFrame(list(kpis.items()), columns=["KPI", "Value"])
display(kpi_df)

# ---------------------------------------------------------------
# 3) Business questions
# ---------------------------------------------------------------

# --- Q1: Which domains generate the most total ARR? ---
arr_by_domain = (
    df.groupby("Domain", sort=False)
    .agg(total_arr=("Revenue_ARR_Millions", "sum"))
    .reset_index()
    .sort_values("total_arr", ascending=False)
)
display(arr_by_domain)

fig = px.bar(
    arr_by_domain, x="Domain", y="total_arr",
    title="Q1 - Total ARR by Domain", text="total_arr",
    color_discrete_sequence=["#636EFA"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Domain", yaxis_title="Total ARR (USD M)")
fig.show()

# --- Q2: How does ARR vary by AI adoption level? ---
arr_by_ai = (
    df.groupby("AI_Adoption_Level", sort=False)
    .agg(total_arr=("Revenue_ARR_Millions", "sum"))
    .reset_index()
    .sort_values("total_arr", ascending=False)
)
display(arr_by_ai)

fig = px.bar(
    arr_by_ai, x="AI_Adoption_Level", y="total_arr",
    title="Q2 - Total ARR by AI Adoption Level", text="total_arr",
    color_discrete_sequence=["#00CC96"],
)
fig.update_layout(xaxis_title="AI Adoption Level", yaxis_title="Total ARR (USD M)")
fig.show()

# --- Q3: How does ARR vary by funding stage? ---
arr_by_stage = (
    df.groupby("Funding_Stage", sort=False)
    .agg(total_arr=("Revenue_ARR_Millions", "sum"))
    .reset_index()
    .sort_values("total_arr", ascending=False)
)
display(arr_by_stage)

fig = px.bar(
    arr_by_stage, x="Funding_Stage", y="total_arr",
    title="Q3 - Total ARR by Funding Stage", text="total_arr",
    color_discrete_sequence=["#FFA15A"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Funding Stage", yaxis_title="Total ARR (USD M)")
fig.show()

# --- Q4: How does ARR vary across countries and cities? ---
arr_by_country = (
    df.groupby("Country", sort=False)
    .agg(total_arr=("Revenue_ARR_Millions", "sum"))
    .reset_index()
    .sort_values("total_arr", ascending=False)
)
display(arr_by_country)

fig = px.bar(
    arr_by_country, x="Country", y="total_arr",
    title="Q4 - Total ARR by Country", text="total_arr",
    color_discrete_sequence=["#AB63FA"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Country", yaxis_title="Total ARR (USD M)")
fig.show()

arr_by_city = (
    df.groupby("City", sort=False)
    .agg(total_arr=("Revenue_ARR_Millions", "sum"))
    .reset_index()
    .sort_values("total_arr", ascending=False)
    .head(10)
)
display(arr_by_city)

fig = px.bar(
    arr_by_city, x="City", y="total_arr",
    title="Q4 - Total ARR by City (Top 10)", text="total_arr",
    color_discrete_sequence=["#AB63FA"],
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="City", yaxis_title="Total ARR (USD M)")
fig.show()

# --- Q5: Is higher funding associated with higher ARR? ---
corr_fa = df["Total_Funding_USD_Millions"].corr(df["Revenue_ARR_Millions"])
print(f"\nCorrelation (funding vs ARR): {corr_fa:.3f}")

fig = px.scatter(
    df, x="Total_Funding_USD_Millions", y="Revenue_ARR_Millions",
    title="Q5 - Funding vs ARR", opacity=0.35,
    labels={
        "Total_Funding_USD_Millions": "Total Funding (USD M)",
        "Revenue_ARR_Millions": "Revenue ARR (USD M)",
    },
    color_discrete_sequence=["#636EFA"],
)
fig.show()

# --- Q6: How are monthly burn and ARR related? ---
corr_ba = df["Monthly_Burn_Rate_Millions"].corr(df["Revenue_ARR_Millions"])
print(f"Correlation (monthly burn vs ARR): {corr_ba:.3f}")

fig = px.scatter(
    df, x="Monthly_Burn_Rate_Millions", y="Revenue_ARR_Millions",
    title="Q6 - Monthly Burn vs ARR", opacity=0.35,
    labels={
        "Monthly_Burn_Rate_Millions": "Monthly Burn (USD M)",
        "Revenue_ARR_Millions": "Revenue ARR (USD M)",
    },
    color_discrete_sequence=["#EF553B"],
)
fig.show()

# --- Q7: Which domains have high ARR relative to monthly burn? ---
domain_burn = df.groupby("Domain", sort=False).agg(
    total_arr=("Revenue_ARR_Millions", "sum"),
    total_burn=("Monthly_Burn_Rate_Millions", "sum"),
).reset_index()
domain_burn["annualized_burn"] = domain_burn["total_burn"] * 12
domain_burn["arr_to_burn"] = domain_burn["total_arr"] / domain_burn["annualized_burn"]
domain_burn = domain_burn.sort_values("total_arr", ascending=False)

display(domain_burn)

domain_burn_melt = domain_burn.melt(
    id_vars="Domain",
    value_vars=["total_arr", "annualized_burn"],
    var_name="metric", value_name="value",
)
domain_burn_melt["metric"] = domain_burn_melt["metric"].map(
    {"total_arr": "ARR", "annualized_burn": "Burn x 12"}
)

fig = px.bar(
    domain_burn_melt, x="Domain", y="value", color="metric", barmode="group",
    title="Q7 - ARR vs Annualized Burn by Domain",
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Domain", yaxis_title="USD M")
fig.show()

# --- Q8: How does revenue-to-burn ratio vary by domain? ---
order = df.groupby("Domain")["Revenue_To_Burn"].median().sort_values().index

fig = px.box(
    df, x="Domain", y="Revenue_To_Burn", category_orders={"Domain": list(order)},
    title="Q8 - Revenue-to-Burn Ratio by Domain (company level)",
)
fig.update_layout(xaxis_tickangle=-45, xaxis_title="Domain", yaxis_title="ARR / (12 x Monthly Burn)")
fig.show()

# --- Q9: Does ARR differ between low and high AI adoption groups? ---
fig = px.box(
    df, x="AI_Adoption_Level", y="Revenue_ARR_Millions",
    category_orders={"AI_Adoption_Level": ["Low", "Medium", "High", "Unknown"]},
    title="Q9 - ARR by AI Adoption Level (box plot)",
)
fig.update_layout(xaxis_title="AI Adoption Level", yaxis_title="Revenue ARR (USD M)")
fig.show()

median_by_ai = df.groupby("AI_Adoption_Level")["Revenue_ARR_Millions"].median().reset_index()
median_by_ai.columns = ["AI_Adoption_Level", "Median_ARR"]
display(median_by_ai.sort_values("Median_ARR", ascending=False))

# --- Q10: Are higher-ARR companies associated with lower layoff rates? ---
df["ARR_Bucket"] = pd.qcut(
    df["Revenue_ARR_Millions"], q=5,
    labels=["Q1 (lowest)", "Q2", "Q3", "Q4", "Q5 (highest)"],
)

arr_layoff = df.groupby("ARR_Bucket", observed=True).agg(
    companies=("Company_ID", "count"),
    median_arr=("Revenue_ARR_Millions", "median"),
    layoff_rate_pct=("Has_Layoffs", "mean"),
).reset_index()
arr_layoff["layoff_rate_pct"] = arr_layoff["layoff_rate_pct"] * 100

display(arr_layoff)

fig = px.bar(
    arr_layoff, x="ARR_Bucket", y="layoff_rate_pct", text="layoff_rate_pct",
    title="Q10 - Layoff Company Rate by ARR Quintile",
    color_discrete_sequence=["#EF553B"],
)
fig.update_layout(xaxis_title="ARR Quintile", yaxis_title="Layoff Company Rate (%)")
fig.show()

,KPI,Value
0,Total Revenue ARR (USD M),832522.26
1,Average ARR per Company (USD M),33.30
2,Median ARR per Company (USD M),4.83
3,Total Monthly Burn (USD M/mo),79221.18
4,Revenue-to-Burn Ratio (total),0.88
5,Total Monthly ARR Proxy Minus Burn (USD M),-9844.33
6,Layoff Company Rate (%),38.84


,Domain,total_arr
4,Generative AI,313447.44
0,HealthTech,90538.13
3,FinTech,78076.72
8,SpaceTech,62985.27
6,ClimateTech,59544.54
1,Web3/Crypto,59354.76
7,Robotics,57596.57
2,Cybersecurity,52729.01
9,Autonomous Vehicles,32912.26
5,EdTech,25337.56


,AI_Adoption_Level,total_arr
0,High,320792.19
1,Medium,264844.15
2,Low,159357.53


,Funding_Stage,total_arr
5,Post-IPO,266262.30
1,Pre-IPO,197731.66
4,Series C+,183445.52
3,Series B,112998.84
2,Series A,50749.27
0,Seed,21334.67


,Country,total_arr
2,USA,391836.59
1,China,118970.61
3,India,77010.83
4,UK,63189.77
6,Germany,42207.09
7,Israel,39398.29
0,Canada,31330.73
5,Singapore,25249.96
9,France,24026.94
8,Japan,19301.45


...


Correlation (funding vs ARR): 0.839


Correlation (monthly burn vs ARR): 0.786


,Domain,total_arr,total_burn,annualized_burn,arr_to_burn
4,Generative AI,313447.44,28507.30,342087.60,0.916278
0,HealthTech,90538.13,9149.00,109788.00,0.824663
3,FinTech,78076.72,6893.27,82719.24,0.943876
8,SpaceTech,62985.27,6032.51,72390.12,0.870081
6,ClimateTech,59544.54,5680.21,68162.52,0.873567
1,Web3/Crypto,59354.76,5919.73,71036.76,0.835550
7,Robotics,57596.57,5772.17,69266.04,0.831527
2,Cybersecurity,52729.01,5056.71,60680.52,0.868961
9,Autonomous Vehicles,32912.26,3282.52,39390.24,0.835544
5,EdTech,25337.56,2927.76,35133.12,0.721187


,AI_Adoption_Level,Median_ARR
1,Low,4.99
0,High,4.91
2,Medium,4.57


,ARR_Bucket,companies,median_arr,layoff_rate_pct
0,Q1 (lowest),5024,0.98,41.560510
1,Q2,4997,2.18,38.483090
2,Q3,4980,4.84,35.823293
3,Q4,5000,13.14,40.180000
4,Q5 (highest),4999,62.98,38.127626
